# Leitura do arquivo `sicor_glebas_wkt_2021.csv`

Este notebook foi preparado para o Google Colab. Ele acessa o arquivo diretamente pelo ID do Google Drive, faz o download autenticado e oferece duas formas de leitura:

1. **Amostra rápida**, para conferir colunas e conteúdo;
2. **Leitura em blocos**, recomendada para o arquivo completo, que possui aproximadamente 656 MB.

> Ao executar a autenticação, use uma conta do Google que tenha permissão para acessar o arquivo.

In [ ]:
# Bibliotecas necessárias (o Colab normalmente já possui todas)
import csv
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

## 1. Autenticar e conectar ao Google Drive

In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

auth.authenticate_user()
credenciais, _ = google.auth.default()
drive = build("drive", "v3", credentials=credenciais)

print("Autenticação concluída.")

## 2. Identificar e baixar o arquivo

O ID abaixo foi extraído do link fornecido. O download é salvo temporariamente no ambiente do Colab, em `/content`.

In [ ]:
from googleapiclient.http import MediaIoBaseDownload

ARQUIVO_ID = "1Ac5ZBLm13_UTnyNtT2Mt2U4sBmrTCjqq"
PASTA_LOCAL = Path("/content/dados_sicor")
PASTA_LOCAL.mkdir(parents=True, exist_ok=True)

metadados = drive.files().get(
    fileId=ARQUIVO_ID,
    fields="id,name,mimeType,size,modifiedTime",
    supportsAllDrives=True,
).execute()

nome_arquivo = metadados["name"]
tamanho_esperado = int(metadados.get("size", 0))
caminho_csv = PASTA_LOCAL / nome_arquivo

print(f"Arquivo: {nome_arquivo}")
print(f"Tipo: {metadados.get('mimeType')}")
print(f"Tamanho: {tamanho_esperado / (1024**2):,.2f} MB")
print(f"Última modificação: {metadados.get('modifiedTime')}")

In [ ]:
# Evita baixar novamente quando já existe uma cópia completa no ambiente.
arquivo_completo = (
    caminho_csv.exists()
    and tamanho_esperado > 0
    and caminho_csv.stat().st_size == tamanho_esperado
)

if arquivo_completo:
    print(f"Arquivo já disponível em: {caminho_csv}")
else:
    requisicao = drive.files().get_media(
        fileId=ARQUIVO_ID,
        supportsAllDrives=True,
    )

    with open(caminho_csv, "wb") as arquivo_local:
        downloader = MediaIoBaseDownload(
            arquivo_local,
            requisicao,
            chunksize=50 * 1024 * 1024,
        )
        concluido = False
        ultimo_percentual = -1

        while not concluido:
            status, concluido = downloader.next_chunk()
            if status:
                percentual = int(status.progress() * 100)
                if percentual != ultimo_percentual:
                    print(f"Download: {percentual}%")
                    ultimo_percentual = percentual

    print(f"Download concluído: {caminho_csv}")

## 3. Detectar codificação e separador

A detecção usa apenas o cabeçalho do CSV. Isso evita que as vírgulas presentes nas geometrias WKT confundam a escolha do separador.

In [ ]:
def detectar_formato_csv(caminho):
    with open(caminho, "rb") as arquivo:
        cabecalho_bytes = arquivo.readline()

    codificacao = None
    cabecalho = None
    for tentativa in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            cabecalho = cabecalho_bytes.decode(tentativa)
            codificacao = tentativa
            break
        except UnicodeDecodeError:
            continue

    if cabecalho is None:
        raise UnicodeError("Não foi possível identificar a codificação do CSV.")

    candidatos = [";", ",", "\t", "|"]
    separador = max(candidatos, key=cabecalho.count)

    if cabecalho.count(separador) == 0:
        raise ValueError("Não foi possível identificar o separador do CSV.")

    return codificacao, separador

CODIFICACAO, SEPARADOR = detectar_formato_csv(caminho_csv)
print(f"Codificação detectada: {CODIFICACAO}")
print(f"Separador detectado: {repr(SEPARADOR)}")

## 4. Fazer uma leitura rápida

Esta célula lê somente as primeiras 100 linhas. Use-a para conferir os nomes das colunas e verificar se o arquivo foi interpretado corretamente.

In [ ]:
df_amostra = pd.read_csv(
    caminho_csv,
    sep=SEPARADOR,
    encoding=CODIFICACAO,
    nrows=100,
    low_memory=False,
)

print(f"Linhas carregadas na amostra: {len(df_amostra):,}")
print(f"Quantidade de colunas: {df_amostra.shape[1]}")
display(df_amostra.head())

In [ ]:
# Relação de colunas e tipos inferidos na amostra
resumo_colunas = pd.DataFrame({
    "coluna": df_amostra.columns,
    "tipo_na_amostra": df_amostra.dtypes.astype(str).values,
    "nulos_na_amostra": df_amostra.isna().sum().values,
})
display(resumo_colunas)

## 5. Processar o arquivo completo em blocos — recomendado

O exemplo abaixo percorre todo o CSV sem manter todas as linhas simultaneamente na memória. Além de contar as linhas, ele calcula o total de valores ausentes por coluna. Você pode acrescentar outras análises dentro do laço.

In [ ]:
TAMANHO_BLOCO = 50_000
total_linhas = 0
nulos_por_coluna = None

leitor = pd.read_csv(
    caminho_csv,
    sep=SEPARADOR,
    encoding=CODIFICACAO,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    total_linhas += len(bloco)
    nulos_bloco = bloco.isna().sum()
    nulos_por_coluna = (
        nulos_bloco
        if nulos_por_coluna is None
        else nulos_por_coluna.add(nulos_bloco, fill_value=0)
    )

    if numero_bloco == 1 or numero_bloco % 10 == 0:
        print(f"Bloco {numero_bloco:,} concluído — {total_linhas:,} linhas processadas")

print(f"Total de linhas do arquivo: {total_linhas:,}")
display(
    nulos_por_coluna
    .astype("int64")
    .rename("quantidade_nulos")
    .to_frame()
    .sort_values("quantidade_nulos", ascending=False)
)

## 6. Leitura completa em um único DataFrame — opcional

Execute somente se houver memória suficiente no Colab. O uso real de memória pode ser várias vezes maior que o tamanho do CSV.

In [ ]:
# Remova o primeiro # das linhas abaixo apenas se quiser carregar tudo na memória.
# df = pd.read_csv(
#     caminho_csv,
#     sep=SEPARADOR,
#     encoding=CODIFICACAO,
#     low_memory=False,
# )
# print(df.shape)
# display(df.head())

## 7. Conferência opcional da coluna de geometria WKT

Caso exista uma coluna chamada `GT_GEOMETRIA`, esta célula mostra apenas o início do texto de algumas geometrias, sem alterar os dados.

In [ ]:
COLUNA_GEOMETRIA = "GT_GEOMETRIA"

if COLUNA_GEOMETRIA in df_amostra.columns:
    geometria_amostra = df_amostra[COLUNA_GEOMETRIA].dropna().astype(str)
    print(f"Geometrias preenchidas na amostra: {len(geometria_amostra)}")
    display(geometria_amostra.str.slice(0, 160).to_frame("inicio_wkt").head())
else:
    print(
        f"A coluna {COLUNA_GEOMETRIA!r} não foi encontrada. "
        "Confira a relação de colunas exibida anteriormente."
    )